# Roxy EDA tutorial on toy protein sequences

This notebook demonstrates a simple exploratory data analysis (EDA) workflow
using **Roxy** on a small toy dataset of protein sequences.

We will:

1. Ensure AAIndex data is available.
2. Build a toy dataset of protein sequences with class labels (`basic` vs `acidic`).
3. Wrap the data into a `RoxyDataset`.
4. Compute global sequence descriptors.
5. Run basic EDA (`BasicEDA`) on the feature block.
6. Build a dataset-level report with feature–target associations.
7. Inspect missingness patterns globally and per label.


In [1]:
import pandas as pd

from roxy.core.dataset import RoxyDataset
from roxy.core.aaindex import load_aaindex, ensure_aaindex_available
from roxy.descriptors.sequences import GlobalSequenceDescriptors

from roxy.eda import (
    BasicEDA,
    build_report,
    missingness_summary,
    missingness_by_label,
)

## 1. Ensure AAIndex data is available

The AAIndex CSV will be downloaded to the Roxy cache directory on first use
if it is not already present. We then inspect the first rows of the table.


In [2]:
# Ensure AAIndex CSV is available (downloads on first call if missing)
ensure_aaindex_available()

aaindex_table = load_aaindex()
aaindex_table.head()

,ANDN920101,ARGP820101,ARGP820102,ARGP820103,BEGF750101,BEGF750102,BEGF750103,BHAR880101,BIGC670101,BIOV880101,...,KARS160113,KARS160114,KARS160115,KARS160116,KARS160117,KARS160118,KARS160119,KARS160120,KARS160121,KARS160122
residue,,,,,,,,,,,,,,,,,,,,,
A,4.35,0.61,1.18,1.56,1.00,0.77,0.37,0.357,52.6,16.0,...,6.0,6.00,6.0,6.0,12.0,6.00,12.000,0.0,6.000,0.000
C,4.65,1.07,1.89,1.23,0.06,0.65,0.84,0.346,68.3,168.0,...,6.0,16.67,12.0,22.0,28.0,9.33,28.000,0.0,11.333,6.000
D,4.76,0.46,0.05,0.14,0.44,0.65,0.97,0.511,68.4,-78.0,...,12.0,16.40,12.0,20.0,34.0,6.80,28.634,0.0,10.400,2.969
E,4.29,0.47,0.11,0.23,0.73,0.55,0.53,0.497,84.7,-106.0,...,12.0,21.00,14.0,26.0,40.0,6.67,28.731,0.0,10.667,1.822
F,4.66,2.02,1.96,2.03,0.60,0.98,0.53,0.314,113.9,189.0,...,18.0,23.25,18.0,24.0,48.0,6.00,26.993,0.0,12.000,2.026


## 2. Build a toy dataset of sequences with labels

We create a small synthetic dataset with 10 sequences. Each sequence contains
all 20 standard amino acids at least once and is labelled as either `basic`
or `acidic` depending on its composition.


In [3]:
toy_data = {
    "sequence": [
        # Each sequence contains all 20 amino acids at least once
        "ACDEFGHIKLMNPQRSTVWYKKKK",   # basic-rich
        "YWVTSRQPNMLKIHGFEDCACRRR",   # basic-rich
        "MNPQRSTVWYACDEFGHIKLKKKR",   # basic-rich
        "DEDEFGHIKLMNPQRSTVWYACDDD",  # acidic-rich
        "CDEFGHIKLMNPQRSTVWYAADEEE",  # acidic-rich
        "ACDGHIKLMNPQRSTVWYEFGRRKK",  # basic-rich
        "ACDEFGHIKLMNPQRSTVWYDDDEE",  # acidic-rich
        "EFGHIKLMNPQRSTVWYACDDDDEE",  # acidic-rich
        "FGHIKLMNPQRSTVWYACDEERRRK",  # basic-rich
        "GHIKLMNPQRSTVWYACDEEFFDDD",  # acidic-rich
    ],
    "label": [
        "basic",
        "basic",
        "basic",
        "acidic",
        "acidic",
        "basic",
        "acidic",
        "acidic",
        "basic",
        "acidic",
    ],
    "id": [
        "prot_1",
        "prot_2",
        "prot_3",
        "prot_4",
        "prot_5",
        "prot_6",
        "prot_7",
        "prot_8",
        "prot_9",
        "prot_10",
    ],
}

samples_df = pd.DataFrame(toy_data).set_index("id")
samples_df

,sequence,label
id,,
prot_1,ACDEFGHIKLMNPQRSTVWYKKKK,basic
prot_2,YWVTSRQPNMLKIHGFEDCACRRR,basic
prot_3,MNPQRSTVWYACDEFGHIKLKKKR,basic
prot_4,DEDEFGHIKLMNPQRSTVWYACDDD,acidic
prot_5,CDEFGHIKLMNPQRSTVWYAADEEE,acidic
prot_6,ACDGHIKLMNPQRSTVWYEFGRRKK,basic
prot_7,ACDEFGHIKLMNPQRSTVWYDDDEE,acidic
prot_8,EFGHIKLMNPQRSTVWYACDDDDEE,acidic
prot_9,FGHIKLMNPQRSTVWYACDEERRRK,basic


## 3. Wrap the samples and labels into a `RoxyDataset`

We keep only the sequence as the raw modality and store the labels and some
metadata about the task.


In [4]:
ds = RoxyDataset(
    samples=samples_df[["sequence"]],   # raw sequence modality
    y=samples_df["label"],              # labels (basic / acidic)
    metadata={"task_type": "classification", "name": "toy_charge_dataset"},
    name="toy_roxy_demo",
)

ds

<RoxyDataset(name='toy_roxy_demo', n_samples=10, n_raw_columns=1)
  feature_blocks: none
  has_y: True>

## 4. Compute global sequence descriptors

We use `GlobalSequenceDescriptors` to compute global features such as
composition, hydrophobicity, charge, etc., and register them under the
feature block key `seq_global`.


In [5]:
# Instantiate the engine with default settings
engine_global = GlobalSequenceDescriptors(pH=7.0, include_histidine_in_charge=False)

# Compute features for all samples
X_global = engine_global.compute(ds.samples)

# Register this block in the dataset under the key 'seq_global'
ds.add_features("seq_global", X_global)

print("Feature blocks:", ds.feature_blocks)
X_global.head()

Feature blocks: ['seq_global']


,length,aa_frac_T,aa_frac_M,aa_frac_S,aa_frac_G,aa_frac_C,aa_frac_R,aa_frac_W,aa_frac_K,aa_frac_N,...,boman_index,net_charge_pH,fcr,ncpr,donors_per_residue,acceptors_per_residue,aa_entropy,lc_k1,lc_k2,lc_k3
_index,,,,,,,,,,,,,,,,,,,,,
prot_1,24.0,0.041667,0.041667,0.041667,0.041667,0.041667,0.041667,0.041667,0.208333,0.041667,...,-0.252917,3.953366,0.375000,0.164724,0.583333,0.333333,4.101227,1.0,0.913043,0.954545
prot_2,24.0,0.041667,0.041667,0.041667,0.041667,0.083333,0.166667,0.041667,0.041667,0.041667,...,-0.197083,2.909937,0.333333,0.121247,0.583333,0.333333,4.168296,1.0,0.956522,1.000000
prot_3,24.0,0.041667,0.041667,0.041667,0.041667,0.041667,0.083333,0.041667,0.166667,0.041667,...,-0.253750,3.953679,0.375000,0.164737,0.583333,0.333333,4.168296,1.0,0.956522,1.000000
prot_4,25.0,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,...,-0.362000,-5.040699,0.400000,-0.201628,0.400000,0.520000,4.099471,1.0,0.916667,1.000000
prot_5,25.0,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,0.040000,...,-0.369200,-4.039321,0.360000,-0.161573,0.400000,0.480000,4.163856,1.0,0.916667,1.000000


## 5. Run basic EDA on the feature block

We now run `BasicEDA` on the `seq_global` feature block to obtain summary
statistics, missingness information and a simple correlation matrix.


In [6]:
eda = BasicEDA(feature_keys=["seq_global"])
basic_report = eda.run(ds)

basic_report.shape, basic_report.feature_keys

((10, 41), ['seq_global'])

In [7]:
# Numeric summary (first few features)
basic_report.numeric_summary.head()

,count,mean,std,min,25%,50%,75%,max
length,10.0,24.7000,0.483046,24.00,24.25,25.00,25.000000,25.000000
aa_frac_T,10.0,0.0405,0.000805,0.04,0.04,0.04,0.041250,0.041667
aa_frac_M,10.0,0.0405,0.000805,0.04,0.04,0.04,0.041250,0.041667
aa_frac_S,10.0,0.0405,0.000805,0.04,0.04,0.04,0.041250,0.041667
aa_frac_G,10.0,0.0445,0.012498,0.04,0.04,0.04,0.041667,0.080000


In [8]:
# Missingness per column (sorted)
basic_report.missing_per_column.sort_values(ascending=False).head()

length       0
aa_frac_T    0
aa_frac_M    0
aa_frac_S    0
aa_frac_G    0
dtype: int64

## 6. Build a dataset-level report with feature–target associations

We combine the selected feature block into a single matrix, then call
`build_report` to obtain a `DatasetReport` including:

- global dataset statistics,
- per-feature summaries,
- univariate feature–target association tests (Kruskal/ANOVA + post hoc),
- correlation matrix.


In [9]:
X_all, y = ds.to_Xy(feature_keys=["seq_global"])
dataset_report = build_report(
    X_all,
    y=y,
    dataset_name=ds.name,
    task_type=ds.metadata.get("task_type", "classification"),
)

dataset_report

DatasetReport(dataset_name='toy_roxy_demo', n_samples=10, n_features=41, task_type='classification', class_distribution={'basic': 5, 'acidic': 5}, feature_summaries={'length': FeatureSummary(name='length', dtype='float64', n_missing=0, missing_ratio=0.0, n_unique=2, mean=24.7, std=0.4830458915396479, min=24.0, max=25.0, skewness=-1.0350983390135262, kurtosis=-1.2244897959183763, target_association={'feature_name': 'length', 'target_name': 'label', 'test_name': 'kruskal', 'statistic': 3.857142857142855, 'p_value': 0.0495346134356268, 'n_groups': 2, 'group_sizes': {'acidic': 5, 'basic': 5}, 'effect_size': 0.3571428571428569, 'posthoc_pvalues': None, 'notes': ['Effect size is epsilon-squared for Kruskal–Wallis.']}), 'aa_frac_T': FeatureSummary(name='aa_frac_T', dtype='float64', n_missing=0, missing_ratio=0.0, n_unique=2, mean=0.040499999999999994, std=0.0008050764858994118, min=0.04, max=0.041666666666666664, skewness=1.0350983390135569, kurtosis=-1.224489795918323, target_association={'f

## 7. Inspect missingness patterns

Finally, we compute missingness diagnostics both globally and stratified
by label using `missingness_summary` and `missingness_by_label`.


In [10]:
miss_global = missingness_summary(X_all)
miss_by_label = missingness_by_label(X_all, y)

miss_global

{'per_column': length                   0
 aa_frac_T                0
 aa_frac_M                0
 aa_frac_S                0
 aa_frac_G                0
 aa_frac_C                0
 aa_frac_R                0
 aa_frac_W                0
 aa_frac_K                0
 aa_frac_N                0
 aa_frac_P                0
 aa_frac_Q                0
 aa_frac_V                0
 aa_frac_L                0
 aa_frac_F                0
 aa_frac_I                0
 aa_frac_E                0
 aa_frac_Y                0
 aa_frac_D                0
 aa_frac_A                0
 aa_frac_H                0
 frac_aromatic            0
 frac_positive            0
 frac_negative            0
 frac_polar               0
 frac_nonpolar            0
 gravy_kd                 0
 hydropathy_eisenberg     0
 top_idp_mean             0
 helix_propensity_mean    0
 sheet_propensity_mean    0
 boman_index              0
 net_charge_pH            0
 fcr                      0
 ncpr                     0
 donor

In [11]:
miss_by_label

{'acidic': {'per_column': length                   0
  aa_frac_T                0
  aa_frac_M                0
  aa_frac_S                0
  aa_frac_G                0
  aa_frac_C                0
  aa_frac_R                0
  aa_frac_W                0
  aa_frac_K                0
  aa_frac_N                0
  aa_frac_P                0
  aa_frac_Q                0
  aa_frac_V                0
  aa_frac_L                0
  aa_frac_F                0
  aa_frac_I                0
  aa_frac_E                0
  aa_frac_Y                0
  aa_frac_D                0
  aa_frac_A                0
  aa_frac_H                0
  frac_aromatic            0
  frac_positive            0
  frac_negative            0
  frac_polar               0
  frac_nonpolar            0
  gravy_kd                 0
  hydropathy_eisenberg     0
  top_idp_mean             0
  helix_propensity_mean    0
  sheet_propensity_mean    0
  boman_index              0
  net_charge_pH            0
  fcr              